# 20 — Export GGUF (CPU-only step)

Prerequisites, checked below and hard-failed if not met:
1. `artifact_gate.py` exits 0 for the checkpoint directory (weights verified against pinned SHA-256).
2. A successful FP16 GPU run report exists (`ok: true`, `mode: benchmark`) from `01_gpu_run.ipynb` — no point exporting a checkpoint that has never produced a real generation.

This step does not need a GPU. It clones llama.cpp, converts the safetensors checkpoint to an f16 GGUF, quantizes to Q4_K_M, and writes a manifest with hashes computed here (never hardcoded).

In [ ]:
import json, os, sys
from pathlib import Path

REPO = Path('/kaggle/working')
sys.path.insert(0, str(REPO))

# Use the actual downloaded model path for 1.5B
CKPT_DIR = '/tmp/modelscope_cache/models/Qwen--Qwen2.5-1.5B-Instruct/snapshots/master'
GPU_REPORT = REPO / 'gpu_run_report.json'  # path used by 01_gpu_run.ipynb

import artifact_gate
gate_ok = artifact_gate.main(str(CKPT_DIR)) if hasattr(artifact_gate, 'main') else None
assert gate_ok is None or gate_ok == 0, 'artifact_gate did not pass — fix that before exporting'

assert GPU_REPORT.exists(), f'missing {GPU_REPORT} — run 01_gpu_run.ipynb first'
report = json.loads(GPU_REPORT.read_text())
assert report.get('ok') is True, 'GPU run report says ok=false — do not export from an unproven checkpoint'
print('Prerequisites satisfied. Proceeding.')

In [ ]:
# Clone llama.cpp. LLAMA_CPP_REF lets you pin a tag; whatever you actually get is
# recorded from git rev-parse, never assumed.
import subprocess

LLAMA_CPP_REF = os.environ.get('LLAMA_CPP_REF', '')  # e.g. 'b4200'; empty = default branch tip
LLAMA_DIR = Path('/kaggle/working/llama.cpp')

if not LLAMA_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1'] +
                           (['--branch', LLAMA_CPP_REF] if LLAMA_CPP_REF else []) +
                           ['https://github.com/ggml-org/llama.cpp', str(LLAMA_DIR)])

llama_cpp_commit = subprocess.check_output(
    ['git', '-C', str(LLAMA_DIR), 'rev-parse', 'HEAD']).decode().strip()
print('llama.cpp commit actually checked out:', llama_cpp_commit)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                        str(LLAMA_DIR / 'requirements' / 'requirements-convert_hf_to_gguf.txt')])
subprocess.check_call(['cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR / 'build'),
                        '-DGGML_CUDA=OFF', '-DLLAMA_CURL=OFF'])
subprocess.check_call(['cmake', '--build', str(LLAMA_DIR / 'build'), '--target',
                        'llama-quantize', '-j', str(os.cpu_count() or 2)])

In [ ]:
# Convert to f16 GGUF, then quantize to Q4_K_M.
OUT_DIR = Path('/kaggle/working/gguf_out')
OUT_DIR.mkdir(exist_ok=True)
F16_PATH = OUT_DIR / 'qwen2.5-1.5b-instruct-f16.gguf'
Q4_PATH = OUT_DIR / 'qwen2.5-1.5b-instruct-Q4_K_M.gguf'

subprocess.check_call([
    sys.executable, str(LLAMA_DIR / 'convert_hf_to_gguf.py'), str(CKPT_DIR),
    '--outfile', str(F16_PATH), '--outtype', 'f16'
])

QUANTIZE_BIN = LLAMA_DIR / 'build' / 'bin' / 'llama-quantize'
subprocess.check_call([str(QUANTIZE_BIN), str(F16_PATH), str(Q4_PATH), 'Q4_K_M'])
print('done:', Q4_PATH, Q4_PATH.stat().st_size, 'bytes')

In [ ]:
# Hash + size everything actually produced. Nothing here is asserted in advance.
import hashlib, time

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    'schema_version': 1,
    'source_model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
    'source_checkpoint_dir': str(CKPT_DIR),
    'llama_cpp_commit': llama_cpp_commit,
    'quant_type': 'Q4_K_M',
    'produced_at': time.time(),
    'files': {
        F16_PATH.name: {'bytes': F16_PATH.stat().st_size, 'sha256': sha256_file(F16_PATH)},
        Q4_PATH.name: {'bytes': Q4_PATH.stat().st_size, 'sha256': sha256_file(Q4_PATH)},
    },
}

manifest_path = OUT_DIR / 'gguf_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print('\nDownload the Q4_K_M gguf + gguf_manifest.json from /kaggle/working/gguf_out and run\n'
      'tools/llama_cpp_local_smoke.py against them on your actual 8GB machine.')